# 13 Control flow

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part III — From commands to scripts</span>
    <span class="bp-meta">Notebook&nbsp;13</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    Letting a script <em>decide</em> and <em>repeat</em> — built, every piece of
    it, on one quiet idea: the exit code each command leaves behind.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. data/ is
# read-only; every script and file we make lives in a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

Your `analyze.sh` from Notebook 12 is a real tool, but it still does one thing,
once. The last step in Part III is to let a script **decide** (do this, not that)
and **repeat** (do it to all of them). That is *control flow*, and it can look like
a pile of unrelated keywords — `if`, `&&`, `while`, `case`, `for`. It is not. They
are all variations on a single idea.

That idea is the **exit code**. Every command, when it finishes, leaves behind a
number: `0` for success, anything else for failure. *All* of control flow is built
on testing that number. `if` tests an exit code; `&&` and `||` chain on it; `while`
loops on it; and `[[ … ]]` is just a command that produces one. Learn to see the
exit code and the rest stops being a list to memorise and becomes one idea, told a
few different ways.

This notebook closes Part III: by the end you can write, edit, and run a tool that
**sweeps a whole directory** and validates its input. (As always — the data is a
playground, **no physics required.**)

## A. Exit codes — the hidden currency

Run a command and the shell records how it went in a special variable, **`$?`**:
`0` means success, any other number means some kind of failure. Two tiny commands
exist purely to demonstrate it — `true` always succeeds, `false` always fails:

In [2]:
true; echo "true gave: $?"

true gave: 0


In [3]:
false; echo "false gave: $?"

false gave: 1


Note the inversion that trips up everyone who comes from C or Python: here **`0` is
success** and non-zero is failure — the opposite of the usual "0 is false." Real
commands follow the same rule; a `grep` that finds nothing "fails" (returns 1):

In [4]:
grep "no such word" data/logs/gr2hno3-nvt.log; echo "grep gave: $?"

grep gave: 1


This is the currency Notebook 12 was already spending: `set -e` watches for a
non-zero exit, and a function's `return` *is* an exit code. Everything below is just
ways of reading `$?` without writing `$?`.

<div class="bp-card">
  <span class="bp-card-cmd">Exit codes</span> — <span class="bp-card-job">the number every command leaves behind. The spine of all control flow.</span>
  <table>
    <tr><td>$?</td><td>the exit code of the last command</td></tr>
    <tr><td>0</td><td><b>success</b> (the opposite of C/Python, where 0 is false)</td></tr>
    <tr><td>1–255</td><td>failure — different non-zero codes can mean different errors</td></tr>
    <tr><td>true&nbsp;&nbsp;/&nbsp;&nbsp;false</td><td>commands that do nothing but succeed / fail — useful in tests</td></tr>
  </table>
</div>

## B. `&&`, `||`, and `;` — chaining on success

The simplest control flow reads exit codes directly. Put two commands together with
**`&&`** and the second runs *only if the first succeeded*; with **`||`**, only if
the first *failed*; with **`;`**, always (just sequence them):

In [5]:
mkdir -p scratch/demo && echo "directory made, so this ran"

directory made, so this ran


The `&&` is the everyday "do this, then that, but only if this worked" — like
`mkdir d && cd d`, which refuses to `cd` into a directory that was not created. Its
mirror, `||`, is the "or else" of error handling:

In [6]:
false || echo "the first command failed, so the fallback ran"

the first command failed, so the fallback ran


<div class="bp-card">
  <span class="bp-card-cmd">Chaining</span> — <span class="bp-card-job">control flow straight on the exit code, no keyword needed.</span>
  <table>
    <tr><td>cmd1 && cmd2</td><td>run cmd2 <b>only if</b> cmd1 succeeded (exit 0)</td></tr>
    <tr><td>cmd1 || cmd2</td><td>run cmd2 <b>only if</b> cmd1 failed (non-zero)</td></tr>
    <tr><td>cmd1 ; cmd2</td><td>just sequence — run cmd2 regardless</td></tr>
    <tr><td>cmd1 | cmd2</td><td>a <i>pipe</i> (Notebook 4) — passes data, not about exit codes</td></tr>
  </table>
</div>

## C. `if` and tests

When the choice needs more than one line, you reach for **`if`**. And here is the
thing to hold onto: `if` does not test a "condition" in the algebra sense — **`if`
runs a command and branches on its exit code.** `if grep -q …; then` runs `grep`
and takes the `then` branch when `grep` succeeded. The shape:

```bash
if   cmd; then
  …            # ran if cmd succeeded
elif cmd2; then
  …            # else, if cmd2 succeeded
else
  …            # otherwise
fi
```

So where do the familiar comparisons (`-gt`, `=`, `-f`) come in? They are not
special syntax — **`[[ … ]]` is itself just a command** whose whole job is to test
something and return `0` or `1`. Watch a numeric test branch:

In [7]:
frames=2520
if [[ "$frames" -gt 1000 ]]; then echo "big run ($frames frames)"; else echo "small run"; fi

big run (2520 frames)


And a file test — the workhorse of input checking — using the data tree:

In [8]:
if [[ -f data/logs/gr2hno3-nvt.log ]]; then echo "the log is there"; fi

the log is there


The operators come in three families, curated to the ones you actually reach for:

<div class="bp-card">
  <span class="bp-card-cmd">Tests for <code>[[ … ]]</code></span> — <span class="bp-card-job">a command that returns 0 (true) or 1 (false). Pick the family that matches your data.</span>
  <table>
    <tr><td>NUMERIC</td><td><code>-eq -ne -lt -le -gt -ge</code> — equal, not-equal, &lt;, ≤, &gt;, ≥</td></tr>
    <tr><td>STRING</td><td><code>=</code> equal, <code>!=</code> not-equal, <code>-z</code> empty, <code>-n</code> non-empty</td></tr>
    <tr><td>FILE</td><td><code>-f</code> a file, <code>-d</code> a directory, <code>-e</code> exists, <code>-r</code>/<code>-w</code>/<code>-x</code> readable/writable/executable</td></tr>
    <tr><td>LOGIC</td><td><code>&amp;&amp;</code> and, <code>||</code> or, <code>!</code> not</td></tr>
    <tr><td><code>=~</code></td><td>matches an extended regex (Notebook 6) — <code>[[ … ]]</code> only</td></tr>
  </table>
</div>

Two notations exist, and the difference matters:

- **`[[ … ]]`** is bash's own, and the one to prefer: it does not word-split unquoted
  variables (Notebook 10), and it understands `&&`, `||`, and `=~` regex inside.
- **`[ … ]`** (a synonym for the `test` command) is the older, POSIX-portable form.
  It works everywhere, but it is fussier — you *must* quote your variables in it.

:::{admonition} ⚠ Inside a test, the spaces are not optional
:class: warning
`[[` and `]]` (and `[` `]`) are *commands*, so they need spaces around them and
around every operator: `[[ "$n" -gt 1000 ]]`, never `[["$n"-gt1000]]`. And mind the
two equals: **`=`** compares *strings*, **`-eq`** compares *numbers* — `[[ "08" =
"8" ]]` is false (different text) while `[[ "08" -eq "8" ]]` is true (same value).
In the portable `[ … ]`, an unquoted empty variable can even break the test's
syntax — which is exactly why `[[ … ]]` is the safer default in bash.
:::

The immediate payoff is **input validation**, the habit that makes a script robust —
and it completes the thought from Notebook 12. A script should check what it was
given before it leans on it: if a required argument is missing, say so and stop.

In [9]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [10]:
cat > scratch/need-arg.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
if [[ $# -lt 1 ]]; then
  echo "usage: need-arg.sh FILE" >&2
  exit 1
fi
echo "working on: $1"
EOF
chmod +x scratch/need-arg.sh
(cd scratch && ./need-arg.sh) 2>&1 || echo "(no argument — it printed usage and exited non-zero)"
(cd scratch && ./need-arg.sh lj38.xyz)

usage: need-arg.sh FILE


(no argument — it printed usage and exited non-zero)


working on: lj38.xyz


No argument, it refuses with a usage line and a non-zero exit; one argument, it
proceeds. That `if [[ $# -lt 1 ]]; then … exit 1; fi` is the single most common
opening of a real-world script.

## D. `case` — pattern dispatch

When you are branching the *same* value against many possibilities, a tall
`if/elif/elif/…` chain gets unreadable. **`case`** is the clean tool for it: give it
a value and a list of **glob patterns** (Notebook 5 — not regex), and it runs the
first arm that matches.

In [11]:
for name in run.log traj.xyz notes.txt; do
  case "$name" in
    *.log)        echo "$name -> a log file" ;;
    *.xyz)        echo "$name -> a trajectory" ;;
    *)            echo "$name -> something else" ;;
  esac
done

run.log -> a log file


traj.xyz -> a trajectory


notes.txt -> something else


Each arm ends with `;;`, alternatives are written `a|b)`, and a final `*)` is the
catch-all default (put it last).

<div class="bp-card">
  <span class="bp-card-cmd">case</span> — <span class="bp-card-job">dispatch a value against glob patterns (Notebook 5) — cleaner than a tall if/elif chain.</span>
  <table>
    <tr><td>case "$x" in</td><td>match <code>$x</code> against the arms below</td></tr>
    <tr><td>&nbsp;&nbsp;pat) … ;;</td><td>run this arm if <code>$x</code> matches the glob <i>pat</i>; end every arm with <code>;;</code></td></tr>
    <tr><td>&nbsp;&nbsp;a|b) … ;;</td><td>alternatives — match <i>a</i> OR <i>b</i></td></tr>
    <tr><td>&nbsp;&nbsp;*) … ;;</td><td>the catch-all default — put it last</td></tr>
    <tr><td>esac</td><td>close the block (<code>case</code> spelled backwards)</td></tr>
  </table>
</div>

Now you have everything to read the piece of
Notebook 12 we deferred — the **`getopts` skeleton**. It was a `while` loop wrapped
around a `case`, and it is just the two tools you now know:

```bash
while getopts "vn:" opt; do   # loop: getopts succeeds while there are flags to read
  case "$opt" in              # dispatch on which flag we got
    v) verbose=1 ;;
    n) count="$OPTARG" ;;
    *) echo "usage: …" >&2; exit 1 ;;
  esac
done
```

That is the whole trick: `getopts` hands one flag per turn to `case`, the `while`
keeps going until the flags run out. Nothing magic — a loop and a switch.

## E. Loops — doing it to all of them

This is the payoff the whole course has been building toward: the recurring "do *X*
to all my files" goal. The everyday loop is **`for`**, and the right thing to loop
over is a **glob** (Notebook 5) — the shell expands it to the matching files, spaces
and all:

In [12]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'a\nb\nc\n' > scratch/run-01.xyz
printf 'a\nb\n' > scratch/run-02.xyz

In [13]:
for f in scratch/*.xyz; do
  echo "$f has $(wc -l < "$f") lines"
done

scratch/run-01.xyz has 3 lines


scratch/run-02.xyz has 2 lines


The loop body ran once per file, the glob kept each name whole. **`for`** also walks
a literal list, a brace sequence `{1..10}`, or a C-style counter; **`while`** and
**`until`** loop on an exit code; and the **`while read`** idiom reads a file line by
line:

<div class="bp-card">
  <span class="bp-card-cmd">Loops</span> — <span class="bp-card-job">repeat a body. <code>break</code> leaves the loop; <code>continue</code> skips to the next turn.</span>
  <table>
    <tr><td>for f in *.xyz; do … done</td><td>iterate a glob, a list, or <code>{1..10}</code></td></tr>
    <tr><td>for ((i=0; i&lt;n; i++)); do … done</td><td>a C-style numeric counter</td></tr>
    <tr><td>while COND; do … done</td><td>loop <b>while</b> an exit code stays 0</td></tr>
    <tr><td>until COND; do … done</td><td>loop <b>until</b> an exit code becomes 0</td></tr>
    <tr><td>while IFS= read -r line<br>&nbsp;&nbsp;do … done &lt; file</td><td>read a file/stream <b>line by line</b>, safely</td></tr>
  </table>
</div>

That last one earns its odd shape. To read a file a line at a time, the safe,
copyable form is exactly:

In [14]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'cutoff 300\nmethod PBE\n' > scratch/params.txt

In [15]:
while IFS= read -r line; do
  echo "saw: $line"
done < scratch/params.txt

saw: cutoff 300


saw: method PBE


The `IFS=` stops leading/trailing spaces from being trimmed and `-r` stops
backslashes from being mangled — together they read each line *verbatim*. (Use
`while read` when the logic is **per line**; when you want **columns** of data, that
is still `awk`'s job, Notebook 8.)

One anti-pattern deserves a hard stop, because it is everywhere and it is the
Notebook-10 word-splitting bug in disguise:

:::{admonition} ⚠ Loop over a glob, never over the output of ls
:class: warning
You will see `for f in $(ls)` in the wild. **Do not write it.** The `$(ls)` is
word-split on spaces, so a file named `my run.xyz` is torn into `my` and `run.xyz`
and your loop runs on names that do not exist. Loop over the **glob** instead —
`for f in *.xyz` — which the shell expands to real, whole filenames. Here is the bug
and its fix, side by side.
:::

In [16]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'x\n' > "scratch/my run.xyz"

In [17]:
for f in $(ls scratch); do echo "[$f]"; done

[my]


[run.xyz]


In [18]:
for f in scratch/*; do echo "[$f]"; done

[scratch/my run.xyz]


The first tore the name in two; the glob kept it intact. Loop over globs.

## F. `seq`, `time`, and a first look at speedup

Two small commands round out the toolkit. **`seq`** prints a sequence of numbers —
for counting and for driving sweeps when the bounds live in variables (where a brace
`{1..10}` cannot reach):

```{command-card} seq
```

In [19]:
seq 0 2 8

0


2


4


6


8


And **`time`** measures how long something takes — prefix any command with it and it
reports the wall-clock `real` time afterwards:

```{command-card} time
```

In [20]:
time sleep 0.2

real	0m0.203s


user	0m0.000s


sys	0m0.002s


Put loops, `seq`, and `time` together and you can ask a question that matters the
moment you reach a cluster: *does throwing more workers at a job actually make it
faster?* Here we run four 0.2-second tasks at one worker, then two — using the
`xargs -P` parallelism alluded to back in Notebook 5, now something you can read:

In [21]:
for w in 1 2; do
  start=$(date +%s.%N)
  seq 4 | xargs -P"$w" -I{} sleep 0.2
  end=$(date +%s.%N)
  awk -v a="$start" -v b="$end" -v w="$w" 'BEGIN { printf "%d worker(s): %.2fs\n", w, b - a }'
done

1 worker(s): 0.81s


2 worker(s): 0.41s


Two workers ran the four tasks in roughly half the wall-clock time of one — a
**speedup of about 2×**. That is the whole game of parallel computing in miniature,
and also its catch: the speedup never keeps doubling forever. Notebook 17 takes this
exact experiment to real scaling data and the question of how many cores are
actually worth asking for.

## Exercises

Control flow is best learned by running it. Everything below works in a fresh
`scratch/`; `data/` stays read-only. (Scripts are written here with here-documents
so the page is reproducible; in your terminal you would type them into Vim.)

### Warm-up 1 (worked) — Exit codes and chaining

Inspect `$?` after a success and a failure, then use `&&` and `||` to act on the
outcome.

In [22]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [23]:
# (solution hidden on the public site)


after true:  0


after false: 1


the log exists


that one does not


In [24]:
ok=$(true; echo $?); bad=$(false; echo $?)
check '[ "$ok" -eq 0 ] && [ "$bad" -ne 0 ]' \
      "true returned 0 (success) and false returned non-zero (failure)"

✓ true returned 0 (success) and false returned non-zero (failure)


### Warm-up 2 (your turn) — A first `if`

Write an `if`/`else` that branches on a **file test**: report whether
`data/logs/gr2hno3-nvt.log` exists, and whether a made-up name does not.

In [25]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [26]:
# (solution hidden on the public site)


log: present


nope: missing


In [27]:
out="$( if [[ -f data/logs/gr2hno3-nvt.log ]]; then echo present; else echo missing; fi )"
check '[ "$out" = "present" ]' \
      "the file test took the correct branch for the file that exists"

✓ the file test took the correct branch for the file that exists


### Applied 1 (your turn) — Validate input

Write a script that requires one argument. If `$#` is zero, print a usage line to
standard error and `exit 1`; otherwise echo the argument. Run it both ways.

In [28]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [29]:
# (solution hidden on the public site)


usage: guard.sh FILE


(refused: no argument)


processing: run.xyz


In [30]:
norc=$(cd "$ROOT/scratch"; ./guard.sh >/dev/null 2>&1; echo $?)
okout="$(cd "$ROOT/scratch" && ./guard.sh run.xyz)"
check '[ "$norc" -eq 1 ] && printf "%s" "$okout" | grep -q "processing: run.xyz"' \
      "it exits 1 with no argument and proceeds when given one"

✓ it exits 1 with no argument and proceeds when given one


### Applied 2 (your turn) — Loop over files

In a scratch directory of `.xyz` files (one with a **space** in its name), loop with
`for f in *.xyz` and print each name with its line count. Do **not** use
`for f in $(ls)`.

In [31]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'a\nb\nc\n' > scratch/run-01.xyz
printf 'a\nb\n'     > scratch/run-02.xyz
printf 'a\n'        > "scratch/run 03.xyz"

In [32]:
# (solution hidden on the public site)


run 03.xyz: 1 lines


run-01.xyz: 3 lines


run-02.xyz: 2 lines


In [33]:
n=$(cd "$ROOT/scratch"; for f in *.xyz; do echo "$f"; done | wc -l)
spaced=$(cd "$ROOT/scratch"; for f in *.xyz; do echo "$f"; done | grep -c "run 03.xyz")
check '[ "$n" -eq 3 ] && [ "$spaced" -eq 1 ]' \
      "the glob loop handled all three files, the spaced name kept whole"

✓ the glob loop handled all three files, the spaced name kept whole


### Applied 3 (worked) — `while read` + `case`

Read a small key/value file line by line and dispatch on the key with `case`.

In [34]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'cutoff 300\nmethod PBE\ncharge 0\n' > scratch/params.txt

In [35]:
# (solution hidden on the public site)


energy cutoff -> cutoff 300


functional    -> method PBE


other          -> charge 0


In [36]:
tags=""
while IFS= read -r line; do
  case "$line" in
    cutoff*) tags="${tags}c" ;;
    method*) tags="${tags}m" ;;
    *)       tags="${tags}o" ;;
  esac
done < scratch/params.txt
check '[ "$tags" = "cmo" ]' \
      "each line was read and dispatched to the right case arm (cutoff, method, other)"

✓ each line was read and dispatched to the right case arm (cutoff, method, other)


### Composite — putting it together (capstone: sweep a directory)

The climax of Part III, and the goal the whole course pointed at. Extend the idea of
`analyze.sh` into `sweep.sh`: it takes a **directory**, validates that argument with
`if`, then **loops over every `.log` in it**, averaging each file's energies with a
function (`grep`/`awk`, Notebooks 6 and 8), and prints a `printf` table of
**file → mean**. One command, a whole directory analysed.

In [37]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/runs
printf ' ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):  -100.000000\n ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):  -102.000000\n' > scratch/runs/run-01.log
printf ' ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):  -120.000000\n ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):  -122.000000\n ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):  -124.000000\n' > scratch/runs/run-02.log
printf ' ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):  -140.000000\n ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):  -141.000000\n' > scratch/runs/run-03.log

In [38]:
# (solution hidden on the public site)


run-01.log   -101.0000


run-02.log   -122.0000


run-03.log   -140.5000


In [39]:
out="$(cd "$ROOT/scratch" && ./sweep.sh runs)"
norc=$(cd "$ROOT/scratch"; ./sweep.sh >/dev/null 2>&1; echo $?)
check '[ "$norc" -eq 1 ] && [ "$(printf "%s\n" "$out" | grep -c "^run-0")" -eq 3 ] && printf "%s" "$out" | grep -q -- "-101.0000" && printf "%s" "$out" | grep -q -- "-122.0000" && printf "%s" "$out" | grep -q -- "-140.5000"' \
      "sweep.sh validated its argument and tabulated the mean of all three logs"

✓ sweep.sh validated its argument and tabulated the mean of all three logs


### Optional stretch (your turn) — Speedup table

Build the speedup teaser into a small table: run a batch of short tasks at **1, 2,
and 4** workers (a `for` loop over the worker counts, `xargs -P"$w"`), timing each
with `date`, and print a row per worker count. Watch the wall time fall — then start
to level off. (Exact times vary; the shape is the point. Notebook 17 develops it.)

In [40]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [41]:
# (solution hidden on the public site)


1 worker(s): 1.62s


2 worker(s): 0.81s


4 worker(s): 0.41s


In [42]:
rows="$(for w in 1 2 4; do start=$(date +%s.%N); seq 8 | xargs -P"$w" -I{} sleep 0.1; end=$(date +%s.%N); awk -v a="$start" -v b="$end" -v w="$w" 'BEGIN{printf "%d worker(s)\n", w}'; done)"
check '[ "$(printf "%s\n" "$rows" | grep -c "worker")" -eq 3 ]' \
      "the table has a timed row for each of the 1, 2, and 4 worker counts"

✓ the table has a timed row for each of the 1, 2, and 4 worker counts


## Outlook

Your scripts can now **decide** and **repeat** — and with that, **Part III is
complete.** You can write a file (Notebook 9), quote it correctly (10), make it
runnable (11), give it inputs and structure (12), and now have it branch, validate,
and sweep an entire directory (13). That is a real command-line tool, built from
nothing.

**Part IV** takes the tool to where the real work happens: the cluster. The
*environment* your scripts run in and how to shape it (Notebook 14); getting onto a
remote machine and moving data to and from it (15); handing your job to a scheduler
that runs it for you (16); and — picking up today's speedup teaser — measuring real
scaling and justifying how many cores to actually ask for (17).

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run every loop and branch yourself — nothing to install. The
    published notebooks ship <b>without worked solutions</b>; if you would like
    the reference solutions — to teach from or to check your own work — get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>